# CourtSide: NBA RAG with local Hugging Face models

This notebook uses the same pipeline as the web app. Select the project `.venv` kernel after running `uv sync --python 3.12 --extra dev`. No API key is needed.


In [ ]:
from nba_rag.config import DATA, LLM_ID, EMBED_ID
from nba_rag.ingest import ingest, chunk_document
from nba_rag.engine import NBAEngine
import json
print("Generator:", LLM_ID, "\nEmbeddings:", EMBED_ID)

## 1. Load official NBA documents
Downloads award and championship history from NBA.com. The first run needs internet.


In [ ]:
if not (DATA / "documents.json").exists():
    ingest()
documents = json.loads((DATA / "documents.json").read_text())
[(doc["title"], doc["url"]) for doc in documents]

## 2. Split into passages
Keep season rows together and attach source provenance to every passage.


In [ ]:
chunks = [chunk for doc in documents for chunk in chunk_document(doc)]
print(len(chunks), "passages")
chunks[0]

## 3. Embed and index locally
MiniLM generates embeddings. Qwen loads in the same process. Weights are cached after the first download.


In [ ]:
engine = NBAEngine()
engine.load()
print(engine.phase, engine.vectors.shape)

## 4. Retrieve evidence
Dense search is reranked by season, keywords, and award category.


In [ ]:
question = "Who won the regular-season MVP in 2023-24?"
passages = engine.retrieve(question)
for passage in passages:
    print(passage["citation"], passage["title"], passage["url"])
    print(passage["text"], "\n")

## 5. Generate a grounded answer
The model is instructed to use only the retrieved passages. Inspect its supporting sources.


In [ ]:
result = engine.ask(question)
print(result["answer"])
print("Generation time:", result["seconds"], "seconds")

## 6. Open the web app
Run this in a terminal from the project directory:

```bash
uv run uvicorn nba_rag.app:app --host 127.0.0.1 --port 8000
```

Open http://localhost:8000 for the chat and expandable source passages. Stop this notebook kernel first on machines with limited RAM, so the models are not loaded twice.
